# Caculate F1 score

In [25]:
from sklearn.metrics import f1_score, accuracy_score
import pickle
import numpy as np
import os

def evaluate_by_batch(loaded_results, batch_size=None):
    attributes = list(loaded_results['predictions'].keys())
    results_summary = {}

    for attr in attributes:
        preds = loaded_results['predictions'][attr].squeeze()
        targets = loaded_results['targets'][attr].squeeze()
        
        # Convert logits to probabilities using sigmoid
        probs = 1.0 / (1.0 + np.exp(-preds))   # → values in [0, 1]
        treshold = 0.5
        if attr == 'Smiling':
            # get from validation set
            treshold = 0.136
        if attr == 'Eyeglasses':
            treshold = 0.159
        #Binarize at 0.5 threshold
        binary_preds = (probs >= treshold).astype(int)
        binary_targets = targets.astype(int)

        if batch_size is None:
            f1 = f1_score(binary_targets, binary_preds)
            acc = accuracy_score(binary_targets, binary_preds)
            results_summary[attr] = {
                'F1-score': round(f1*100, 3),
                'Accuracy': round(acc*100, 3)
            }
        else:
            f1_list, acc_list = [], []
            for i in range(0, len(binary_targets), batch_size):
                batch_preds = binary_preds[i:i+batch_size]
                batch_targets = binary_targets[i:i+batch_size]

                if len(np.unique(batch_targets)) > 1:
                    f1 = f1_score(batch_targets, batch_preds)
                else:
                    f1 = 1.0 if (batch_targets == batch_preds).all() else 0.0

                acc = accuracy_score(batch_targets, batch_preds)
                f1_list.append(f1)
                acc_list.append(acc)

            f1_mean, f1_std = np.mean(f1_list), np.std(f1_list)
            acc_mean, acc_std = np.mean(acc_list), np.std(acc_list)

            results_summary[attr] = {
                'F1-score': round(f1_mean, 3),
                'F1-std': round(f1_std, 2),
                'Accuracy': round(acc_mean, 3),
                'Acc-std': round(acc_std, 2)
            }

    return results_summary


# === 设置评估目标属性 ===
batch_size = None

# === 根目录 ===
root_dir = "../saved_benchmark/celebahq_simple/newcls_DSCM_effectiveness_step100.0_scale1.5_BlendTrue"
'''SD3'''
#root_dir = "../saved_benchmark/celebahq_simple/200000_newcls_DSCM_effectiveness_step50.0_scale1.7_NTIFalse_2025-10-28T20-32-57-controlnet_textcond_constrastive_7attrsgeneration_text_global_after/"
root_dir = "../saved_benchmark/celebahq_simple/100000_newcls_DSCM_effectiveness_step50.0_scale1.8_NTIFalse_2025-10-28T20-32-57-controlnet_textcond_constrastive_7attrsgeneration_text_global_after_SRC1.5/"
# === 遍历所有 do(attr) 文件夹 ===
do_folders = [name for name in ["Smiling","Eyeglasses"] if os.path.isdir(os.path.join(root_dir, name))]


for target_attr in ["Smiling","Eyeglasses"]:
    print(f"Comparing '{batch_size}' F1-score on attribute '{target_attr}' under different do(.) conditions:")
    for do_attr in do_folders:
        result_path = os.path.join(root_dir, do_attr, "final_results.pkl")
        if not os.path.exists(result_path):
            continue  # skip missing

        with open(result_path, "rb") as f:
            loaded_results = pickle.load(f)

        results = evaluate_by_batch(loaded_results, batch_size=batch_size)

        if target_attr not in results:
            continue  # skip if target attr not in result

        scores = results[target_attr]
        if batch_size is None:
            print(f"do({do_attr})\t{target_attr}: F1 = {scores['F1-score']}, Acc = {scores['Accuracy']}")
        else:
            print(f"do({do_attr})\t{target_attr}: F1 = {scores['F1-score']} ± {scores['F1-std']}, Acc = {scores['Accuracy']} ± {scores['Acc-std']}")


Comparing 'None' F1-score on attribute 'Smiling' under different do(.) conditions:
do(Smiling)	Smiling: F1 = 94.712, Acc = 94.547
do(Eyeglasses)	Smiling: F1 = 98.193, Acc = 98.3
Comparing 'None' F1-score on attribute 'Eyeglasses' under different do(.) conditions:
do(Smiling)	Eyeglasses: F1 = 99.713, Acc = 99.965
do(Eyeglasses)	Eyeglasses: F1 = 97.394, Acc = 95.22


In [12]:

for do_attr in do_folders:
    result_path = os.path.join(root_dir, do_attr, "final_results.pkl")
    if not os.path.exists(result_path):
        continue  # skip missing

    with open(result_path, "rb") as f:
        loaded_results = pickle.load(f)

    print('do {} lipips distance is {}'.format(do_attr,np.mean(loaded_results['lpips'])))

do Smiling lipips distance is 0.03542875498533249
do Eyeglasses lipips distance is 0.059643056243658066


In [17]:
import os, pickle
import numpy as np
from sklearn.metrics import roc_auc_score

def sigmoid(x):
    # numerically stable sigmoid
    x = np.clip(x, -80, 80)
    return 1.0 / (1.0 + np.exp(-x))

best_thresholds = {}  # {(target_attr, do_attr): {'thr_f1':..., 'f1':..., 'thr_acc':..., 'acc':..., 'auroc':...}}

for target_attr in ["Smiling", "Eyeglasses"]:
    print(f"\n=== Attribute: {target_attr} ===")
    for do_attr in do_folders:
        result_path = os.path.join(root_dir, do_attr, "final_results.pkl")
        if not os.path.exists(result_path):
            continue

        with open(result_path, "rb") as f:
            loaded_results = pickle.load(f)

        # ---- get logits & targets for the current target_attr ----
        preds = np.asarray(loaded_results['predictions'][target_attr]).squeeze()
        targets = np.asarray(loaded_results['targets'][target_attr]).squeeze().astype(int)

        # ensure 1-D
        preds = preds.reshape(-1)
        targets = targets.reshape(-1)

        # drop NaNs/Infs if any
        mask = np.isfinite(preds) & np.isfinite(targets)
        preds, targets = preds[mask], targets[mask]

        # logits -> probs
        probs = sigmoid(preds)

        # scan thresholds
        thresholds = np.linspace(0.0, 1.0, 1001)
        P = targets.astype(bool)

        f1_list, acc_list, prec_list, rec_list = [], [], [], []
        for t in thresholds:
            pred = probs >= t
            TP = np.sum(pred & P)
            FP = np.sum(pred & ~P)
            FN = np.sum(~pred & P)
            TN = np.sum(~pred & ~P)

            precision = TP / (TP + FP) if (TP + FP) else 0.0
            recall    = TP / (TP + FN) if (TP + FN) else 0.0
            f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
            acc       = (TP + TN) / (TP + FP + FN + TN) if (TP + FP + FN + TN) else 0.0

            f1_list.append(f1)
            acc_list.append(acc)
            prec_list.append(precision)
            rec_list.append(recall)

        f1_arr  = np.asarray(f1_list)
        acc_arr = np.asarray(acc_list)

        best_f1_idx  = int(np.argmax(f1_arr))
        best_acc_idx = int(np.argmax(acc_arr))

        best_f1_thr  = thresholds[best_f1_idx]
        best_acc_thr = thresholds[best_acc_idx]

        # AUROC (optional, threshold-independent)
        auroc = None
        try:
            auroc = roc_auc_score(targets, probs)
        except Exception:
            pass

        print(f"[{do_attr}]  Best F1  @ thr={best_f1_thr:.3f}: "
              f"F1={f1_arr[best_f1_idx]:.4f}, Precision={prec_list[best_f1_idx]:.4f}, Recall={rec_list[best_f1_idx]:.4f}")
        print(f"[{do_attr}]  Best Acc @ thr={best_acc_thr:.3f}: Acc={acc_arr[best_acc_idx]:.4f}"
              + (f" | AUROC={auroc:.4f}" if auroc is not None else ""))

        best_thresholds[(target_attr, do_attr)] = {
            'thr_f1': float(best_f1_thr), 'f1': float(f1_arr[best_f1_idx]),
            'thr_acc': float(best_acc_thr), 'acc': float(acc_arr[best_acc_idx]),
            'auroc': None if auroc is None else float(auroc)
        }

        # If you want final binarization using best F1 threshold:
        final_threshold = best_f1_thr
        binary_preds = (probs >= final_threshold).astype(int)



=== Attribute: Smiling ===
[Smiling]  Best F1  @ thr=0.974: F1=0.9631, Precision=0.9605, Recall=0.9657
[Smiling]  Best Acc @ thr=0.974: Acc=0.9610 | AUROC=0.9894
[Eyeglasses]  Best F1  @ thr=0.665: F1=0.9699, Precision=0.9659, Recall=0.9739
[Eyeglasses]  Best Acc @ thr=0.665: Acc=0.9713 | AUROC=0.9965

=== Attribute: Eyeglasses ===
[Smiling]  Best F1  @ thr=0.999: F1=1.0000, Precision=1.0000, Recall=1.0000
[Smiling]  Best Acc @ thr=0.999: Acc=1.0000 | AUROC=1.0000
[Eyeglasses]  Best F1  @ thr=0.001: F1=0.9949, Precision=0.9970, Recall=0.9928
[Eyeglasses]  Best Acc @ thr=0.001: Acc=0.9904 | AUROC=0.9954


# Find threshold from validation set

In [46]:
import random
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score, f1_score

import sys
sys.path.append("../../../")
sys.path.append("/home/jovyan/fcvm-data-volume/kzzr229/workspace/counterfactual-benchmark/counterfactual_benchmark")
sys.path.append('/home/jovyan/fcvm-data-volume/kzzr229/workspace/MCPL-diffuser')

from edit_modules.load_celebahq import CelebAHQ
from models.classifiers.celeba_classifier import CelebaClassifier,Celeba_anticausal_Classifier

# ------------------------------
# 1. Load test dataset
# ------------------------------
def dataset_load_path(data_root, dataset, split='train'):
    data_dir = data_root
    data = CelebAHQ(root=data_dir, split=split, transform=None, download=False)
    num_images = len(data)
    if 'simple' in dataset:
        selected_item = ['Smiling', 'Eyeglasses', 'Mouth_Slightly_Open', 'Male', 'Bald', 'Wearing_Lipstick', 'Wearing_Hat']
    elif 'complex' in dataset:
        raise NotImplementedError('complex dataset not handled here')
    else:
        raise AssertionError(f'no such {dataset} dataset')

    attribute_ids = [data.attr_names.index(attr) for attr in selected_item]
    metrics = {attr: torch.as_tensor(data.attr[:, attr_id], dtype=torch.float32)
               for attr, attr_id in zip(selected_item, attribute_ids)}
    attrs = torch.cat([metrics[attr].unsqueeze(1) for attr in selected_item], dim=1)
    possible_values = {attr: torch.unique(values, dim=0) for attr, values in metrics.items()}
    return data, attrs, num_images

data_root = '/home/jovyan/fcvm-data-volume/kzzr229/workspace/counterfactual-benchmark/datasets/'
dataset = 'simple'
data, _, num_images = dataset_load_path(data_root, dataset, split='valid')
#ori_data, imglabel, _ = dataset_load_path(data_root, dataset, split='test')

In [51]:
import os, pickle
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.metrics import f1_score, accuracy_score

# --- your project imports ---
import sys
sys.path += [
    "/home/jovyan/fcvm-data-volume/kzzr229/workspace/counterfactual-benchmark/",
    "/home/jovyan/fcvm-data-volume/kzzr229/workspace/counterfactual-benchmark/counterfactual_benchmark",
    "/home/jovyan/fcvm-data-volume/kzzr229/workspace/MCPL-diffuser-flux",
]
from models.classifiers.celeba_classifier import Celeba_anticausal_Classifier

device = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------
# 1) Validation dataset & loader
# ------------------------------
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    # If used during training, enable Normalize with the same stats:
    # transforms.Normalize([0.5, 0.5, 0.5],[0.5, 0.5, 0.5]),
])

# Adjust args to your CelebAHQ initializer if different
data.transform = transform
val_loader = DataLoader(data, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

# Attribute indices
attr_idx_smile = data.attr_names.index("Smiling")
attr_idx_glass = data.attr_names.index("Eyeglasses")

# ------------------------------
# 2) Load classifiers
# ------------------------------
# Smiling
smile_cls = CelebaClassifier(attr="Smiling", num_outputs=1)
smile_ckpt = '/home/jovyan/fcvm-data-volume/kzzr229/workspace/counterfactual-benchmark/counterfactual_benchmark/methods/deepscm/checkpoints/celebahq/simple/trained_classifiers/backed/Smiling_classifier-epoch=13.ckpt'

#smile_cls = Celeba_anticausal_Classifier(attr="Smiling", num_outputs=1, pretrain=False).to(device).eval()
#smile_ckpt = "/home/jovyan/fcvm-data-volume/kzzr229/workspace/counterfactual-benchmark/counterfactual_benchmark/methods/deepscm/checkpoints/celebahq/simple/trained_classifiers/backed1/Smiling_classifier-epoch=35-val_metric=0.9880.ckpt"


smile_state = torch.load(smile_ckpt, map_location=device)["state_dict"]
smile_cls.load_state_dict(smile_state)
smile_cls.cuda()

# Eyeglasses
glass_cls = CelebaClassifier(attr="Eyeglasses", num_outputs=1)
glass_ckpt = '/home/jovyan/fcvm-data-volume/kzzr229/workspace/counterfactual-benchmark/counterfactual_benchmark/methods/deepscm/checkpoints/celebahq/simple/trained_classifiers/backed/Eyeglasses_classifier-epoch=14.ckpt'

#glass_cls = Celeba_anticausal_Classifier(attr="Eyeglasses", num_outputs=1, pretrain=False).to(device).eval()
#glass_ckpt = "/home/jovyan/fcvm-data-volume/kzzr229/workspace/counterfactual-benchmark/counterfactual_benchmark/methods/deepscm/checkpoints/celebahq/simple/trained_classifiers/backed1/Eyeglasses_classifier-epoch=18-val_metric=0.9896.ckpt"
glass_state = torch.load(glass_ckpt, map_location=device)["state_dict"]
glass_cls.load_state_dict(glass_state)
glass_cls.cuda()
# ------------------------------
# 3) Utilities
# ------------------------------
def collect_probs_targets(model, attr_idx, loader, device="cuda"):
    """Run model on loader and return concatenated (probs, targets) numpy arrays."""
    all_probs, all_tgts = [], []
    model.eval()
    with torch.no_grad():
        for images, attrs in loader:
            images = images.to(device, non_blocking=True)
            logits = model(images).squeeze(1)              # [B]
            probs  = torch.sigmoid(logits).float().cpu().numpy()
            tgts   = attrs[:, attr_idx].long().cpu().numpy()
            all_probs.append(probs)
            all_tgts.append(tgts)
    return np.concatenate(all_probs), np.concatenate(all_tgts)

def best_threshold_by_f1(probs, targets, step=0.001):
    """Scan thresholds in [0,1] to maximize F1. Returns (best_thr, best_f1, acc_at_best)."""
    thresholds = np.arange(0.0, 1.0 + 1e-12, step)
    best_f1, best_thr, best_acc = -1.0, 0.5, 0.0
    for t in thresholds:
        preds = (probs >= t).astype(int)
        f1 = f1_score(targets, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(t)
            best_acc = accuracy_score(targets, preds)
    return best_thr, best_f1, best_acc

# ------------------------------
# 4) Compute thresholds on validation set
# ------------------------------
# Smiling
probs_smile, tgts_smile = collect_probs_targets(smile_cls, attr_idx_smile, val_loader, device=device)
best_thr_smile, best_f1_smile, acc_at_smile = best_threshold_by_f1(probs_smile, tgts_smile, step=0.001)

# Eyeglasses
probs_glass, tgts_glass = collect_probs_targets(glass_cls, attr_idx_glass, val_loader, device=device)
best_thr_glass, best_f1_glass, acc_at_glass = best_threshold_by_f1(probs_glass, tgts_glass, step=0.001)

# ------------------------------
# 5) Report
# ------------------------------
print("\n=== Best validation thresholds (per classifier) ===")
print(f"Smiling   -> best_thr={best_thr_smile:.3f} | F1={best_f1_smile:.4f} | Acc={acc_at_smile:.4f}")
print(f"Eyeglasses-> best_thr={best_thr_glass:.3f} | F1={best_f1_glass:.4f} | Acc={acc_at_glass:.4f}")

# If you want to use these later during test:
best_thresholds = {"Smiling": best_thr_smile, "Eyeglasses": best_thr_glass}


=== Best validation thresholds (per classifier) ===
Smiling   -> best_thr=0.155 | F1=0.9235 | Acc=0.9285
Eyeglasses-> best_thr=0.922 | F1=0.9426 | Acc=0.9953
